# Практика 3 — Преобразование и JOIN — профиль организаций HMDA

## Подготовка

In [11]:
import json
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.2 (6830, 7)


## 01. Входы, строка и ключ

In [12]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei":"string"})
assert len(organizations) == 470
assert not organizations.duplicated(org_keys).any()
assert not organizations[org_keys].isna().any().any()
assert not decisions[org_keys].isna().any().any()
organizations.head(4)

,lei,organization_name,source_count,activity_year
0,549300AG64NHILB7ZP05,"LOANDEPOT.COM, LLC",114,2023
1,QFROUN1UWUYU0DVIWD51,"Fifth Third Bank, National Association",3,2023
2,5493003GQDUH26DNNH17,NAVY FEDERAL CREDIT UNION,742,2023
3,549300DD5QQUHO6PCH70,"Mortgage Research Center, LLC",83,2023


## 01. Входы, строка и ключ — шаг 2

In [13]:
long = decisions.groupby(long_keys, dropna=False).size().rename("n").reset_index()
print(long.shape, long["n"].sum())
assert not long.duplicated(long_keys).any()
assert long["n"].sum() == len(decisions)
long.head(6)

(500, 4) 6830


,lei,activity_year,action_taken,n
0,03D0JEWFDFUS0SEEKG89,2023,1,76
1,03D0JEWFDFUS0SEEKG89,2023,3,15
2,1IE8VN30JCEQV1H4R804,2023,1,5
3,1IE8VN30JCEQV1H4R804,2023,3,2
4,213800KVD76WKI6PN868,2023,1,1
5,213800KVD76WKI6PN868,2023,3,1


## A. Завершите разворот

In [14]:
wide_task = long.pivot(index=org_keys, columns="action_taken", values="n")

print("Размер wide_task:", wide_task.shape)
print("Имена уровней индекса:", wide_task.index.names)
print("Число пропусков по каждому исходу:\n", wide_task.isna().sum())


Размер wide_task: (314, 3)
Имена уровней индекса: ['lei', 'activity_year']
Число пропусков по каждому исходу:
 action_taken
1     20
2    252
3    170
dtype: int64


## 02. Обоснованный ноль и явные колонки

In [15]:
wide = long.pivot(index=org_keys, columns="action_taken", values="n")
wide = wide.fillna(0).astype("int64")
wide = wide.rename(columns={1:"issued", 2:"approved_not_accepted", 3:"denied"})
wide = wide.rename_axis(None, axis=1).reset_index()
value_cols = ["issued", "approved_not_accepted", "denied"]
assert wide[value_cols].sum().sum() == 6830
wide.head(4)

,lei,activity_year,issued,approved_not_accepted,denied
0,03D0JEWFDFUS0SEEKG89,2023,76,0,15
1,1IE8VN30JCEQV1H4R804,2023,5,0,2
2,213800KVD76WKI6PN868,2023,1,0,1
3,213800QUAI2VH5YM6310,2023,76,4,9


## 02. Обоснованный ноль и явные колонки — шаг 2

In [16]:
restored = wide.melt(id_vars=org_keys, value_vars=value_cols, var_name="outcome", value_name="n")
print(restored.shape)
restored.head(6)

(942, 4)


,lei,activity_year,outcome,n
0,03D0JEWFDFUS0SEEKG89,2023,issued,76
1,1IE8VN30JCEQV1H4R804,2023,issued,5
2,213800KVD76WKI6PN868,2023,issued,1
3,213800QUAI2VH5YM6310,2023,issued,76
4,213800XR2TCBQJSF1X93,2023,issued,12
5,254900114YHTRFBALC74,2023,issued,6


## B. Восстановите наблюдаемые комбинации

In [17]:
outcome_to_code = {
    "issued": 1,
    "approved_not_accepted": 2,
    "denied": 3,
}

restored_task = restored.copy()
restored_task["action_taken"] = restored_task["outcome"].map(outcome_to_code)
assert restored_task["action_taken"].notna().all(), "Обнаружены нераспознанные имена исходов"

restored_task = (
    restored_task.loc[restored_task["n"] > 0, long_keys + ["n"]]
    .sort_values(long_keys)
    .reset_index(drop=True)
)
long_sorted = long.sort_values(long_keys).reset_index(drop=True)

assert_frame_equal(restored_task, long_sorted)

print("Всего развёрнутых сочетаний (wide -> melt):", len(restored))
print("Наблюдаемых сочетаний (n > 0):", len(restored_task))
print("Исключено нулевых сочетаний:", len(restored) - len(restored_task))
print("Сумма n:", restored_task["n"].sum())


Всего развёрнутых сочетаний (wide -> melt): 942
Наблюдаемых сочетаний (n > 0): 500
Исключено нулевых сочетаний: 442
Сумма n: 6830


## 03. Добавляем имя с контрактом

In [18]:
lookup = organizations[org_keys + ["organization_name"]].copy()
enriched = decisions.merge(lookup, on=org_keys, how="left", validate="many_to_one", indicator=True)
print(enriched.shape)
print(enriched["_merge"].value_counts())
assert len(enriched) == 6830
assert enriched["source_row"].is_unique
assert set(enriched["source_row"]) == set(decisions["source_row"])
assert enriched["_merge"].eq("both").all()
enriched[["source_row", "lei", "organization_name", "action_taken"]].head(5)

(6830, 9)
_merge
both          6830
left_only        0
right_only       0
Name: count, dtype: int64


,source_row,lei,organization_name,action_taken
0,2,549300VZVN841I2ILS84,"CROSSCOUNTRY MORTGAGE, LLC",1
1,3,549300P1347BFCBHJV60,MASS. INSTITUTE OF TECH. FEDERAL CREDIT UNION,1
2,5,549300P1347BFCBHJV60,MASS. INSTITUTE OF TECH. FEDERAL CREDIT UNION,1
3,7,549300RPOGWJRH63HS39,UNION HOME MORTGAGE CORP.,1
4,8,549300RPOGWJRH63HS39,UNION HOME MORTGAGE CORP.,1


## C. Проверьте сохранность самих значений

In [19]:
orig_cols = decisions.columns.tolist()

left_check = decisions.sort_values("source_row").reset_index(drop=True)
right_check = enriched[orig_cols].sort_values("source_row").reset_index(drop=True)
assert_frame_equal(left_check, right_check)
print("Проверка исходных 7 полей пройдена успешно.")

corrupted_enriched = enriched.copy()
valid_idx = corrupted_enriched["loan_amount"].first_valid_index()
corrupted_enriched.loc[valid_idx, "loan_amount"] = corrupted_enriched.loc[valid_idx, "loan_amount"] + 1000

try:
    assert_frame_equal(
        left_check,
        corrupted_enriched[orig_cols].sort_values("source_row").reset_index(drop=True),
    )
except AssertionError:
    print("Ожидаемо: assert_frame_equal обнаружил изменение в loan_amount.")
else:
    raise RuntimeError("Ошибка: проверка не сработала на изменённых данных!")


Проверка исходных 7 полей пройдена успешно.
Ожидаемо: assert_frame_equal обнаружил изменение в loan_amount.


## 04. Неполная доставка: потеря при inner

In [20]:
partial_lookup = lookup.iloc[1:].copy()
left_partial = decisions.merge(partial_lookup, on=org_keys, how="left", validate="many_to_one", indicator=True)
inner_partial = decisions.merge(partial_lookup, on=org_keys, how="inner", validate="many_to_one")
print(len(left_partial), len(inner_partial))
print(left_partial["_merge"].value_counts())
assert len(left_partial) == 6830
assert len(inner_partial) == 6805

6830 6805
_merge
both          6805
left_only       25
right_only       0
Name: count, dtype: int64


## 04. Неполная доставка: потеря при inner — шаг 2

In [21]:
replayed_lookup = pd.concat([lookup, lookup.iloc[[0]]], ignore_index=True)
wrong_join = decisions.merge(replayed_lookup, on=org_keys, how="left")
print("Без validate:", len(wrong_join))
assert len(wrong_join) == 6855
try:
    decisions.merge(replayed_lookup, on=org_keys, how="left", validate="many_to_one")
except pd.errors.MergeError:
    print("Повторный правый ключ остановлен")
else:
    raise AssertionError("Ожидалась остановка JOIN")

Без validate: 6855
Повторный правый ключ остановлен


## D. Сохраните аудит покрытия

In [22]:
unmatched = left_partial.loc[
    left_partial["_merge"] == "left_only",
    ["source_row", "lei", "activity_year", "action_taken"],
].copy()
unmatched.to_csv(WORK / "unmatched.csv", index=False)

lost_in_inner = set(decisions["source_row"]) - set(inner_partial["source_row"])
assert lost_in_inner == set(unmatched["source_row"]), "Расхождение между left_only и потерями при inner"

join_audit = {
    "left_rows": int(len(decisions)),
    "right_rows": int(len(partial_lookup)),
    "result_rows": int(len(left_partial)),
    "unmatched_rows": int(len(unmatched)),
    "expected_relation": "many_to_one",
    "note": "Учебная неполная копия справочника partial_lookup (исключена первая организация)",
}

with open(WORK / "join_audit.json", "w", encoding="utf-8") as f:
    json.dump(join_audit, f, ensure_ascii=False, indent=2)

print("Сохранено unmatched.csv, строк:", len(unmatched))
print("Аудит соединения:", join_audit)


Сохранено unmatched.csv, строк: 25
Аудит соединения: {'left_rows': 6830, 'right_rows': 469, 'result_rows': 6830, 'unmatched_rows': 25, 'expected_relation': 'many_to_one', 'note': 'Учебная неполная копия справочника partial_lookup (исключена первая организация)'}


## 05. Профиль с собственным знаменателем

In [23]:
prepared = decisions.copy()
prepared["is_approved"] = prepared["action_taken"].isin([1,2])
profile = prepared.groupby(org_keys).agg(n_decisions=("action_taken","size"), n_approved=("is_approved","sum")).reset_index()
profile["approval_rate"] = profile["n_approved"] / profile["n_decisions"]
named_profile = profile.merge(lookup, on=org_keys, how="left", validate="one_to_one")
assert len(named_profile) == 314
assert named_profile["n_decisions"].sum() == 6830
assert named_profile["n_approved"].sum() == 6139
named_profile.sort_values("n_decisions", ascending=False).head(5)

,lei,activity_year,n_decisions,n_approved,approval_rate,organization_name
119,549300CF8MP6S7MZV277,2023,420,413,0.983333,FIRST SAVINGS MORTGAGE CORPORATION
147,549300GPO6DWUZR4UY30,2023,402,379,0.942786,FIRST HOME MORTGAGE CORPORATION
290,E57ODZWZ7FF32TWEFA76,2023,247,239,0.967611,"Citibank, National Association"
81,5493005PKOSG7MYX0B34,2023,235,230,0.978723,"Prosperity Home Mortgage, LLC"
296,JJKC32MCHWDI71265Z06,2023,216,192,0.888889,Truist Bank


## E. Передайте проверяемый результат

In [24]:
profile_path = WORK / "organization_profile.csv"
named_profile.to_csv(profile_path, index=False)

reloaded_profile = pd.read_csv(profile_path, dtype={"lei": "string"})
assert_frame_equal(named_profile, reloaded_profile)

weighted_approval = named_profile["n_approved"].sum() / named_profile["n_decisions"].sum()
unweighted_approval = named_profile["approval_rate"].mean()

print("Профиль успешно сохранён и проверен при повторном чтении:", reloaded_profile.shape)
print(f"Общая доля одобрений (взвешенная): {weighted_approval:.4f}")
print(f"Простое среднее approval_rate по организациям: {unweighted_approval:.4f}")


Профиль успешно сохранён и проверен при повторном чтении: (314, 6)
Общая доля одобрений (взвешенная): 0.8988
Простое среднее approval_rate по организациям: 0.8578
